# Practical 12 — Customer Segmentation using K-Means and Hierarchical Clustering

Standalone solution notebook for **ML Practical 12**. Run cells from top to bottom and place the required dataset CSV in the working directory.https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, mean_squared_error, mean_absolute_error,
    r2_score, roc_auc_score
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

---
# Practical 12 — Customer Segmentation using K-Means and Hierarchical Clustering
**Aim:** Apply K-Means and hierarchical clustering to a customer sales dataset and use the elbow method to select the number of clusters.

The assignment describes customer ID, gender, age, annual income, and spending score. Common Kaggle files use `Mall_Customers.csv` with columns such as `CustomerID`, `Gender`, `Age`, `Annual Income (k$)`, and `Spending Score (1-100)`. The code also accepts `sales_data_sample.csv` if it has compatible features.

In [ ]:
CLUSTER_PATH = "Mall_Customers.csv"
if not os.path.exists(CLUSTER_PATH):
    alternatives = ["sales_data_sample.csv", "Sales_data_sample.csv", "mall_customers.csv"]
    found = next((f for f in alternatives if os.path.exists(f)), None)
    if found:
        CLUSTER_PATH = found
    else:
        raise FileNotFoundError(
            "Customer dataset not found. Put Mall_Customers.csv or sales_data_sample.csv beside this notebook."
        )

customers = pd.read_csv(CLUSTER_PATH)
customers.columns = [str(c).strip() for c in customers.columns]
print("Shape:", customers.shape)
display(customers.head())
print("Columns:", customers.columns.tolist())

In [ ]:
# Detect annual income, spending score, and age columns with flexible names.
def find_column(columns, candidates):
    lookup = {str(c).strip().lower().replace("_", " "): c for c in columns}
    for candidate in candidates:
        key = candidate.lower().replace("_", " ")
        if key in lookup:
            return lookup[key]
    for c in columns:
        cl = str(c).strip().lower()
        if any(term in cl for term in candidates):
            return c
    return None

income_col = find_column(customers.columns, ["annual income (k$)", "annual income", "income"])
spending_col = find_column(customers.columns, ["spending score (1-100)", "spending score", "spending"])
age_col = find_column(customers.columns, ["age", "customer age"])

if income_col is None or spending_col is None:
    raise ValueError(
        "Could not detect annual-income and spending-score columns. "
        "This practical requires a customer segmentation dataset with these features."
    )

feature_cols = [income_col, spending_col]
if age_col is not None:
    feature_cols = [age_col, income_col, spending_col]

cluster_data = customers[feature_cols].apply(pd.to_numeric, errors="coerce").dropna()
print("Features used:", feature_cols, "| rows:", len(cluster_data))
display(cluster_data.describe())

In [ ]:
# Scale features so the clustering distance is not dominated by one column.
cluster_scaler = StandardScaler()
X_cluster = cluster_scaler.fit_transform(cluster_data)

from sklearn.cluster import KMeans, AgglomerativeClustering
wcss = []
k_values = range(1, 11)
for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_cluster)
    wcss.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), wcss, marker="o")
plt.xticks(list(k_values))
plt.xlabel("Number of clusters (K)")
plt.ylabel("WCSS / Inertia")
plt.title("Elbow Method for Choosing K")
plt.show()

In [ ]:
# Choose K based on the elbow plot. For the common Mall Customers dataset, 5 is a reasonable starting point.
N_CLUSTERS = 5
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans.fit_predict(X_cluster)

clustered = cluster_data.copy()
clustered["KMeans_Cluster"] = kmeans_labels

if income_col in clustered.columns and spending_col in clustered.columns:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(
        data=clustered, x=income_col, y=spending_col,
        hue="KMeans_Cluster", palette="tab10", s=60
    )
    plt.title(f"K-Means Customer Segments (K={N_CLUSTERS})")
    plt.show()

display(clustered.head(10))
print("K-Means cluster counts:")
display(clustered["KMeans_Cluster"].value_counts().sort_index())

In [ ]:
# Hierarchical (Agglomerative) clustering using the same selected number of clusters.
hierarchical = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage="ward")
hierarchical_labels = hierarchical.fit_predict(X_cluster)

clustered["Hierarchical_Cluster"] = hierarchical_labels
print("Hierarchical cluster counts:")
display(clustered["Hierarchical_Cluster"].value_counts().sort_index())

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=clustered, x=income_col, y=spending_col,
    hue="Hierarchical_Cluster", palette="tab10", s=60
)
plt.title(f"Hierarchical Customer Segments (K={N_CLUSTERS})")
plt.show()

**Result:** The elbow curve helps estimate K; K-Means and hierarchical clustering then assign customer groups. The final cluster count should be chosen after inspecting the elbow plot and the structure of the actual dataset.